In [ ]:
# =============================================================================
# NOTEBOOK: transform_partner_raw_data_v2_final.py
# VERSION  : v5 — Performance & Memory Optimized
#
# THAY ĐỔI SO VỚI v4:
#   1. CELL 5 & 12: Dọn dẹp duplicate imports & duplicate helper functions.
#   2. CELL 7  : Persist `validated` trước khi chia `latest_valid`, `all_valid_toast`
#                và `reject_df` -> Giảm từ 3 lần scan bảng raw xuống 1 lần scan duy nhất.
#   3. CELL 10 : Thay `table_state_df.count() == 0` bằng `isEmpty()`, loại bỏ count() thừa.
#   4. CELL 10 : Bỏ gọi lại `ensure_target_table_exists()` trùng lặp trong `process_table()`.
#   5. CELL 10 & 12: Giữ State dưới dạng Spark DataFrames (`state_dfs_all`) thay vì `.collect()`
#                hàng trăm ngàn dicts về Python Driver memory -> loại bỏ rủi ro Driver OOM
#                và triệt tiêu 100% PySpark serialization overhead.
# =============================================================================


# %%
# =============================================================================
# CELL 1 — IMPORT VÀ CẤU HÌNH CHẠY
# =============================================================================

from datetime import datetime, timedelta
import traceback
import uuid

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, StructType, StructField,
    StringType, TimestampType, LongType, IntegerType
)
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

PIPELINE_NAME = "transform_partner_raw_data_v2"
RUN_ID        = str(uuid.uuid4())

RAW_TABLE = "lh_vv_bronze.dbo.partner_raw_data"

WATERMARK_TABLE = "lh_vv_bronze.partner.brz_watermark"
WATERMARK_ID    = "wm_transform_partner_raw_data"

RUN_AUDIT_TABLE   = "lh_vv_bronze.partner.cdc_run_audit"
TABLE_AUDIT_TABLE = "lh_vv_bronze.partner.cdc_table_run_audit"
REJECT_TABLE      = "lh_vv_bronze.partner.partner_cdc_reject"
CDC_STATE_TABLE   = "lh_vv_bronze.partner.cdc_record_state"

RAW_INGEST_COLUMN   = "EventProcessedUtcTime"
RAW_SEQUENCE_COLUMN = None

WATERMARK_OVERLAP_MINUTES = 10

VALID_OPS      = ["c", "r", "u", "d"]
UPSERT_OPS     = ["c", "r", "u"]
TOAST_SENTINEL = "__debezium_unavailable_value"

DEFAULT_WATERMARK  = datetime(2000, 1, 1)
DEFAULT_SEQUENCE   = -1
TARGET_TABLE_COUNT = 23

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")
# Tối ưu shuffle cho F8/F16: giảm số partition để tránh overhead
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("=" * 60)
print(f"PIPELINE         : {PIPELINE_NAME}")
print(f"RUN_ID           : {RUN_ID}")
print(f"RAW TABLE        : {RAW_TABLE}")
print(f"RAW_INGEST_COLUMN: {RAW_INGEST_COLUMN}")
print(f"WATERMARK TABLE  : {WATERMARK_TABLE}")
print("=" * 60)


# %%
# =============================================================================
# CELL 2 — KHAI BÁO 23 BẢNG (TABLE_CONFIGS)
# =============================================================================

TABLE_CONFIGS = [
    {
        "source_table": "products",
        "target_table": "lh_vv_bronze.partner.products",
        "key": "id",
        "columns": {
            "id":            {"path": "id",            "type": "string"},
            "external_id":   {"path": "external_id",   "type": "string"},
            "product_line":  {"path": "product_line",  "type": "string"},
            "status":        {"path": "status",        "type": "string"},
            "attributes":    {"path": "attributes",    "type": "string"},
            "metadata":      {"path": "metadata",      "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "created_at":    {"path": "created_at",    "type": "bigint"},
            "updated_at":    {"path": "updated_at",    "type": "bigint"},
            "deleted_at":    {"path": "deleted_at",    "type": "string"},
        },
        "toast_columns": ["attributes", "metadata", "raw_sync_data"],
    },
    {
        "source_table": "product_translations",
        "target_table": "lh_vv_bronze.partner.product_translations",
        "key": "id",
        "columns": {
            "id":         {"path": "id",         "type": "bigint"},
            "product_id": {"path": "product_id", "type": "string"},
            "locale":     {"path": "locale",     "type": "string"},
            "name":       {"path": "name",       "type": "string"},
            "metadata":   {"path": "metadata",   "type": "string"},
            "created_at": {"path": "created_at", "type": "bigint"},
            "updated_at": {"path": "updated_at", "type": "bigint"},
        },
        "toast_columns": ["metadata"],
    },
    {
        "source_table": "product_attributes",
        "target_table": "lh_vv_bronze.partner.product_attributes",
        "key": "id",
        "columns": {
            "id":                     {"path": "id",                     "type": "string"},
            "external_id":            {"path": "external_id",            "type": "string"},
            "code":                   {"path": "code",                   "type": "string"},
            "status":                 {"path": "status",                 "type": "string"},
            "product_category_scope": {"path": "product_category_scope", "type": "string"},
            "raw_sync_data":          {"path": "raw_sync_data",          "type": "string"},
            "created_at":             {"path": "created_at",             "type": "bigint"},
            "updated_at":             {"path": "updated_at",             "type": "bigint"},
        },
        "toast_columns": ["raw_sync_data"],
    },
    {
        "source_table": "product_attribute_translations",
        "target_table": "lh_vv_bronze.partner.product_attribute_translations",
        "key": "id",
        "columns": {
            "id":           {"path": "id",           "type": "bigint"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "locale":       {"path": "locale",       "type": "string"},
            "name":         {"path": "name",         "type": "string"},
            "created_at":   {"path": "created_at",   "type": "bigint"},
            "updated_at":   {"path": "updated_at",   "type": "bigint"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_attribute_values",
        "target_table": "lh_vv_bronze.partner.product_attribute_values",
        "key": "id",
        "columns": {
            "id":            {"path": "id",            "type": "string"},
            "attribute_id":  {"path": "attribute_id",  "type": "string"},
            "external_id":   {"path": "external_id",   "type": "string"},
            "code":          {"path": "code",          "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "created_at":    {"path": "created_at",    "type": "bigint"},
            "updated_at":    {"path": "updated_at",    "type": "bigint"},
        },
        "toast_columns": ["raw_sync_data"],
    },
    {
        "source_table": "product_attribute_value_translations",
        "target_table": "lh_vv_bronze.partner.product_attribute_value_translations",
        "key": "id",
        "columns": {
            "id":           {"path": "id",           "type": "bigint"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "locale":       {"path": "locale",       "type": "string"},
            "name":         {"path": "name",         "type": "string"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_variants",
        "target_table": "lh_vv_bronze.partner.product_variants",
        "key": "id",
        "columns": {
            "id":            {"path": "id",            "type": "string"},
            "product_id":    {"path": "product_id",    "type": "string"},
            "status":        {"path": "status",        "type": "string"},
            "external_id":   {"path": "external_id",   "type": "string"},
            "attributes":    {"path": "attributes",    "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "created_at":    {"path": "created_at",    "type": "bigint"},
            "updated_at":    {"path": "updated_at",    "type": "bigint"},
            "deleted_at":    {"path": "deleted_at",    "type": "string"},
        },
        "toast_columns": ["attributes", "raw_sync_data"],
    },
    {
        "source_table": "product_variant_translations",
        "target_table": "lh_vv_bronze.partner.product_variant_translations",
        "key": "id",
        "columns": {
            "id":         {"path": "id",         "type": "bigint"},
            "variant_id": {"path": "variant_id", "type": "string"},
            "locale":     {"path": "locale",     "type": "string"},
            "name":       {"path": "name",       "type": "string"},
            "created_at": {"path": "created_at", "type": "bigint"},
            "updated_at": {"path": "updated_at", "type": "bigint"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_business_service_relation",
        "target_table": "lh_vv_bronze.partner.product_business_service_relation",
        "key": "id",
        "columns": {
            "id":                  {"path": "id",                  "type": "bigint"},
            "product_id":          {"path": "product_id",          "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "created_at":          {"path": "created_at",          "type": "bigint"},
            "updated_at":          {"path": "updated_at",          "type": "bigint"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_posts",
        "target_table": "lh_vv_bronze.partner.product_posts",
        "key": "id",
        "columns": {
            "id":                  {"path": "id",                  "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "created_by":          {"path": "created_by",          "type": "string"},
            "updated_by":          {"path": "updated_by",          "type": "string"},
            "post_code":           {"path": "post_code",           "type": "string"},
            "external_id":         {"path": "external_id",         "type": "string"},
            "status":              {"path": "status",              "type": "string"},
            "original_price":      {"path": "original_price",      "type": "string"},
            "markup_percent":      {"path": "markup_percent",      "type": "string"},
            "marketing_price":     {"path": "marketing_price",     "type": "string"},
            "thumbnail_url":       {"path": "thumbnail_url",       "type": "string"},
            "gallery":             {"path": "gallery",             "type": "string"},
            "product_list":        {"path": "product_list",        "type": "string"},
            "raw_sync_data":       {"path": "raw_sync_data",       "type": "string"},
            "validity_from":       {"path": "validity_from",       "type": "bigint"},
            "validity_to":         {"path": "validity_to",         "type": "bigint"},
            "created_at":          {"path": "created_at",          "type": "bigint"},
            "updated_at":          {"path": "updated_at",          "type": "bigint"},
            "deleted_at":          {"path": "deleted_at",          "type": "string"},
        },
        "toast_columns": [
            "original_price", "markup_percent", "marketing_price",
            "gallery", "product_list", "raw_sync_data",
        ],
    },
    {
        "source_table": "product_post_translations",
        "target_table": "lh_vv_bronze.partner.product_post_translations",
        "key": "id",
        "columns": {
            "id":                {"path": "id",                "type": "bigint"},
            "product_post_id":   {"path": "product_post_id",   "type": "string"},
            "locale":            {"path": "locale",            "type": "string"},
            "title":             {"path": "title",             "type": "string"},
            "short_description": {"path": "short_description", "type": "string"},
            "tags":              {"path": "tags",              "type": "string"},
            "filter_tags":       {"path": "filter_tags",       "type": "string"},
            "ai_tags":           {"path": "ai_tags",           "type": "string"},
            "faqs":              {"path": "faqs",              "type": "string"},
            "sections":          {"path": "sections",          "type": "string"},
            "meta_title":        {"path": "meta_title",        "type": "string"},
            "meta_description":  {"path": "meta_description",  "type": "string"},
            "slug":              {"path": "slug",              "type": "string"},
            "created_at":        {"path": "created_at",        "type": "bigint"},
            "updated_at":        {"path": "updated_at",        "type": "bigint"},
        },
        "toast_columns": ["tags", "filter_tags", "ai_tags", "faqs", "sections"],
    },
    {
        "source_table": "partners",
        "target_table": "lh_vv_bronze.partner.partners",
        "key": "id",
        "columns": {
            "id":                            {"path": "id",                            "type": "string"},
            "code":                          {"path": "code",                          "type": "string"},
            "lead_id":                       {"path": "lead_id",                       "type": "string"},
            "company_name":                  {"path": "company_name",                  "type": "string"},
            "company_short_name":            {"path": "company_short_name",            "type": "string"},
            "company_legal_name":            {"path": "company_legal_name",            "type": "string"},
            "company_legal_vietnamese_name": {"path": "company_legal_vietnamese_name", "type": "string"},
            "company_tax_id":                {"path": "company_tax_id",                "type": "string"},
            "company_registration_number":   {"path": "company_registration_number",   "type": "string"},
            "company_address":               {"path": "company_address",               "type": "string"},
            "business_sector":               {"path": "business_sector",               "type": "string"},
            "business_sector_detail":        {"path": "business_sector_detail",        "type": "string"},
            "detailed_profile":              {"path": "detailed_profile",              "type": "string"},
            "status":                        {"path": "status",                        "type": "string"},
            "assignee_username":             {"path": "assignee_username",             "type": "string"},
            "operation_status":              {"path": "operation_status",              "type": "string"},
            "created_at":                    {"path": "created_at",                    "type": "bigint"},
            "last_updated_at":               {"path": "last_updated_at",               "type": "bigint"},
            "created_by":                    {"path": "created_by",                    "type": "string"},
            "last_updated_by":               {"path": "last_updated_by",               "type": "string"},
            "tier":                          {"path": "tier",                          "type": "smallint"},
            "metadata":                      {"path": "metadata",                      "type": "string"},
            "parent_partner_id":             {"path": "parent_partner_id",             "type": "string"},
            "type":                          {"path": "type",                          "type": "string"},
            "business_scenario":             {"path": "business_scenario",             "type": "string"},
            "contact_information":           {"path": "contact_information",           "type": "string"},
            "business_registration_link":    {"path": "business_registration_link",    "type": "string"},
            "legal_representative":          {"path": "legal_representative",          "type": "string"},
            "business_registration_change":  {"path": "business_registration_change",  "type": "string"},
            "headquarter_information":       {"path": "headquarter_information",       "type": "string"},
        },
        "toast_columns": [
            "detailed_profile", "metadata", "contact_information",
            "legal_representative", "business_registration_change", "headquarter_information",
        ],
    },
    {
        "source_table": "business_services",
        "target_table": "lh_vv_bronze.partner.business_services",
        "key": "id",
        "columns": {
            "id":                   {"path": "id",                   "type": "string"},
            "partner_id":           {"path": "partner_id",           "type": "string"},
            "business_location_id": {"path": "business_location_id", "type": "string"},
            "name":                 {"path": "name",                 "type": "string"},
            "phone":                {"path": "phone",                "type": "string"},
            "hotline":              {"path": "hotline",              "type": "string"},
            "email":                {"path": "email",                "type": "string"},
            "business_sector":      {"path": "business_sector",      "type": "string"},
            "operation_status":     {"path": "operation_status",     "type": "string"},
            "address":              {"path": "address",              "type": "string"},
            "direction_note":       {"path": "direction_note",       "type": "string"},
            "bio":                  {"path": "bio",                  "type": "string"},
            "story":                {"path": "story",                "type": "string"},
            "usps":                 {"path": "usps",                 "type": "string"},
            "local_tips":           {"path": "local_tips",           "type": "string"},
            "media":                {"path": "media",                "type": "string"},
            "is_main":              {"path": "is_main",              "type": "boolean"},
            "amentities":           {"path": "amentities",           "type": "string"},
            "created_at":           {"path": "created_at",           "type": "bigint"},
            "last_updated_at":      {"path": "last_updated_at",      "type": "bigint"},
            "created_by":           {"path": "created_by",           "type": "string"},
            "last_updated_by":      {"path": "last_updated_by",      "type": "string"},
            "metadata":             {"path": "metadata",             "type": "string"},
            "is_active":            {"path": "is_active",            "type": "boolean"},
            "last_in_activated_at": {"path": "last_in_activated_at", "type": "bigint"},
            "last_in_activated_by": {"path": "last_in_activated_by", "type": "string"},
            "website_urls":         {"path": "website_urls",         "type": "string"},
            "last_verify_at":       {"path": "last_verify_at",       "type": "epoch_seconds_timestamp"},
        },
        "toast_columns": [
            "address", "direction_note", "bio", "story", "usps",
            "local_tips", "media", "amentities", "metadata", "website_urls",
        ],
    },
    {
        "source_table": "business_locations",
        "target_table": "lh_vv_bronze.partner.business_locations",
        "key": "id",
        "columns": {
            "id":                {"path": "id",                "type": "string"},
            "name":              {"path": "name",              "type": "string"},
            "type":              {"path": "type",              "type": "string"},
            "address":           {"path": "address",           "type": "string"},
            "direction_note":    {"path": "direction_note",    "type": "string"},
            "partner_id":        {"path": "partner_id",        "type": "string"},
            "business_scenario": {"path": "business_scenario", "type": "string"},
            "created_at":        {"path": "created_at",        "type": "bigint"},
            "last_updated_at":   {"path": "last_updated_at",   "type": "bigint"},
            "last_updated_by":   {"path": "last_updated_by",   "type": "string"},
            "metadata":          {"path": "metadata",          "type": "string"},
            "created_by":        {"path": "created_by",        "type": "string"},
        },
        "toast_columns": ["address", "direction_note", "metadata"],
    },
    {
        "source_table": "hotel_room_types",
        "target_table": "lh_vv_bronze.partner.hotel_room_types",
        "key": "id",
        "columns": {
            "id":                   {"path": "id",                   "type": "string"},
            "code":                 {"path": "code",                 "type": "string"},
            "name":                 {"path": "name",                 "type": "string"},
            "description":          {"path": "description",          "type": "string"},
            "status":               {"path": "status",               "type": "string"},
            "max_occupancy":        {"path": "max_occupancy",        "type": "int"},
            "adult_capacity":       {"path": "adult_capacity",       "type": "int"},
            "child_capacity":       {"path": "child_capacity",       "type": "int"},
            "max_extra_bed":        {"path": "max_extra_bed",        "type": "int"},
            "max_child_surcharge":  {"path": "max_child_surcharge",  "type": "int"},
            "room_size":            {"path": "room_size",            "type": "int"},
            "view_type":            {"path": "view_type",            "type": "string"},
            "total_physical_rooms": {"path": "total_physical_rooms", "type": "int"},
            "total_pseudo_rooms":   {"path": "total_pseudo_rooms",   "type": "int"},
            "out_of_service_count": {"path": "out_of_service_count", "type": "int"},
            "out_of_order_count":   {"path": "out_of_order_count",   "type": "int"},
            "total_rooms":          {"path": "total_rooms",          "type": "int"},
            "available_rooms":      {"path": "available_rooms",      "type": "int"},
            "amenities":            {"path": "amenities",            "type": "string"},
            "media_files":          {"path": "media_files",          "type": "string"},
            "tags":                 {"path": "tags",                 "type": "string"},
            "business_service_id":  {"path": "business_service_id",  "type": "string"},
            "created_at":           {"path": "created_at",           "type": "bigint"},
            "updated_at":           {"path": "updated_at",           "type": "bigint"},
            "deleted_at":           {"path": "deleted_at",           "type": "string"},
        },
        "toast_columns": ["description", "amenities", "media_files", "tags"],
    },
    {
        "source_table": "hotel_rate_plans",
        "target_table": "lh_vv_bronze.partner.hotel_rate_plans",
        "key": "id",
        "columns": {
            "id":                     {"path": "id",                     "type": "string"},
            "business_service_id":    {"path": "business_service_id",    "type": "string"},
            "code":                   {"path": "code",                   "type": "string"},
            "name":                   {"path": "name",                   "type": "string"},
            "description":            {"path": "description",            "type": "string"},
            "status":                 {"path": "status",                 "type": "string"},
            "kind":                   {"path": "kind",                   "type": "string"},
            "meal_plan":              {"path": "meal_plan",              "type": "int"},
            "include_tax":            {"path": "include_tax",            "type": "boolean"},
            "include_service_charge": {"path": "include_service_charge", "type": "boolean"},
            "is_derived":             {"path": "is_derived",             "type": "boolean"},
            "valid_from":             {"path": "valid_from",             "type": "bigint"},
            "valid_to":               {"path": "valid_to",               "type": "bigint"},
            "is_visible_on_website":  {"path": "is_visible_on_website",  "type": "boolean"},
            "tags":                   {"path": "tags",                   "type": "string"},
            "cancellation_policies":  {"path": "cancellation_policies",  "type": "string"},
            "created_at":             {"path": "created_at",             "type": "bigint"},
            "updated_at":             {"path": "updated_at",             "type": "bigint"},
            "deleted_at":             {"path": "deleted_at",             "type": "string"},
        },
        "toast_columns": ["description", "tags", "cancellation_policies"],
    },
    {
        "source_table": "business_service_i18ns",
        "target_table": "lh_vv_bronze.partner.business_service_i18ns",
        "key": ["business_service_id", "locale"],
        "columns": {
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "locale":              {"path": "locale",              "type": "string"},
            "name":                {"path": "name",                "type": "string"},
            "bio":                 {"path": "bio",                 "type": "string"},
            "story":               {"path": "story",               "type": "string"},
            "direction_note":      {"path": "direction_note",      "type": "string"},
            "usps":                {"path": "usps",                "type": "string"},
            "local_tips":          {"path": "local_tips",          "type": "string"},
            "amentities":          {"path": "amentities",          "type": "string"},
            "media":               {"path": "media",               "type": "string"},
            "created_at":          {"path": "created_at",          "type": "bigint"},
            "updated_at":          {"path": "updated_at",          "type": "bigint"},
        },
        "toast_columns": [
            "name", "bio", "story", "direction_note",
            "usps", "local_tips", "amentities", "media",
        ],
    },
    {
        "source_table": "business_service_poi_link",
        "target_table": "lh_vv_bronze.partner.business_service_poi_link",
        "key": "id",
        "columns": {
            "id":                   {"path": "id",                   "type": "bigint"},
            "business_service_id":  {"path": "business_service_id",  "type": "string"},
            "poi_id":               {"path": "poi_id",               "type": "string"},
            "created_at":           {"path": "created_at",           "type": "bigint"},
            "updated_at":           {"path": "updated_at",           "type": "bigint"},
            "lang":                 {"path": "lang",                 "type": "string"},
        },
    },
    {
        "source_table": "orders",
        "target_table": "lh_vv_bronze.partner.orders",
        "key": "id",
        "columns": {
            "id":             {"path": "id",             "type": "string"},
            "code":           {"path": "code",           "type": "string"},
            "contact_info":   {"path": "contact_info",   "type": "string"},
            "user_id":        {"path": "user_id",        "type": "string"},
            "total_payment":  {"path": "total_payment",  "type": "decimal(19,4)"},
            "currency":       {"path": "currency",       "type": "string"},
            "status":         {"path": "status",         "type": "string"},
            "payment_status": {"path": "payment_status", "type": "string"},
            "payment_id":     {"path": "payment_id",     "type": "string"},
            "metadata":       {"path": "metadata",       "type": "string"},
            "expire_at":      {"path": "expire_at",      "type": "bigint"},
            "created_at":     {"path": "created_at",     "type": "bigint"},
            "updated_at":     {"path": "updated_at",     "type": "bigint"},
        },
        "toast_columns": ["contact_info", "metadata"],
    },
    {
        "source_table": "order_refs",
        "target_table": "lh_vv_bronze.partner.order_refs",
        "key": "id",
        "columns": {
            "id":                    {"path": "id",                    "type": "string"},
            "order_id":              {"path": "order_id",              "type": "string"},
            "status":                {"path": "status",                "type": "string"},
            "total_payment":         {"path": "total_payment",         "type": "decimal(19,4)"},
            "sub_total":             {"path": "sub_total",             "type": "decimal(19,4)"},
            "external_order_id":     {"path": "external_order_id",     "type": "string"},
            "internal_input_object": {"path": "internal_input_object", "type": "string"},
            "product_type":          {"path": "product_type",          "type": "string"},
            "raw_data":              {"path": "raw_data",              "type": "string"},
            "created_at":            {"path": "created_at",            "type": "bigint"},
            "updated_at":            {"path": "updated_at",            "type": "bigint"},
        },
        "toast_columns": ["internal_input_object", "raw_data"],
    },
    {
        "source_table": "order_item_tickets",
        "target_table": "lh_vv_bronze.partner.order_item_tickets",
        "key": "id",
        "columns": {
            "id":                           {"path": "id",                           "type": "string"},
            "order_id":                     {"path": "order_id",                     "type": "string"},
            "order_ref_id":                 {"path": "order_ref_id",                 "type": "string"},
            "product_id":                   {"path": "product_id",                   "type": "string"},
            "variant_id":                   {"path": "variant_id",                   "type": "string"},
            "provider":                     {"path": "provider",                     "type": "string"},
            "pnr":                          {"path": "pnr",                          "type": "string"},
            "ticket_number":                {"path": "ticket_number",                "type": "string"},
            "verify_code":                  {"path": "verify_code",                  "type": "string"},
            "external_ticket_product_name": {"path": "external_ticket_product_name", "type": "string"},
            "usage_date":                   {"path": "usage_date",                   "type": "timestamp"},
            "valid_from":                   {"path": "valid_from",                   "type": "timestamp"},
            "valid_to":                     {"path": "valid_to",                     "type": "timestamp"},
            "raw_data":                     {"path": "raw_data",                     "type": "string"},
            "created_at":                   {"path": "created_at",                   "type": "bigint"},
            "updated_at":                   {"path": "updated_at",                   "type": "bigint"},
        },
        "toast_columns": ["raw_data"],
    },
    {
        "source_table": "order_item_hotels",
        "target_table": "lh_vv_bronze.partner.order_item_hotels",
        "key": "id",
        "columns": {
            "id":             {"path": "id",             "type": "string"},
            "order_id":       {"path": "order_id",       "type": "string"},
            "order_ref_id":   {"path": "order_ref_id",   "type": "string"},
            "check_in_date":  {"path": "check_in_date",  "type": "timestamp"},
            "checkout_date":  {"path": "checkout_date",  "type": "timestamp"},
            "number_of_room": {"path": "number_of_room", "type": "smallint"},
            "hotel_id":       {"path": "hotel_id",       "type": "string"},
            "pnr":            {"path": "pnr",            "type": "string"},
            "room_type_id":   {"path": "room_type_id",   "type": "string"},
            "rate_plan_id":   {"path": "rate_plan_id",   "type": "string"},
            "raw_data":       {"path": "raw_data",       "type": "string"},
            "created_at":     {"path": "created_at",     "type": "bigint"},
            "updated_at":     {"path": "updated_at",     "type": "bigint"},
        },
        "toast_columns": ["raw_data"],
    },
    {
        "source_table": "order_item_flights",
        "target_table": "lh_vv_bronze.partner.order_item_flights",
        "key": "id",
        "columns": {
            "id":                 {"path": "id",                 "type": "string"},
            "order_id":           {"path": "order_id",           "type": "string"},
            "order_ref_id":       {"path": "order_ref_id",       "type": "string"},
            "provider":           {"path": "provider",           "type": "string"},
            "pnr":                {"path": "pnr",                "type": "string"},
            "booking_id":         {"path": "booking_id",         "type": "string"},
            "segment_id":         {"path": "segment_id",         "type": "smallint"},
            "carrier_code":       {"path": "carrier_code",       "type": "string"},
            "carrier_name":       {"path": "carrier_name",       "type": "string"},
            "departure":          {"path": "departure",          "type": "string"},
            "departure_time":     {"path": "departure_time",     "type": "timestamp"},
            "departure_terminal": {"path": "departure_terminal", "type": "string"},
            "arrival":            {"path": "arrival",            "type": "string"},
            "arrival_time":       {"path": "arrival_time",       "type": "timestamp"},
            "arrival_terminal":   {"path": "arrival_terminal",   "type": "string"},
            "flight_number":      {"path": "flight_number",      "type": "string"},
            "booking_class":      {"path": "booking_class",      "type": "string"},
            "aircraft":           {"path": "aircraft",           "type": "string"},
            "fare_basis_code":    {"path": "fare_basis_code",    "type": "string"},
            "fare_basis_detail":  {"path": "fare_basis_detail",  "type": "string"},
            "raw_data":           {"path": "raw_data",           "type": "string"},
            "created_at":         {"path": "created_at",         "type": "bigint"},
            "updated_at":         {"path": "updated_at",         "type": "bigint"},
        },
        "toast_columns": ["fare_basis_detail", "raw_data"],
    },
]

assert len(TABLE_CONFIGS) == TARGET_TABLE_COUNT
_src_names = [c["source_table"] for c in TABLE_CONFIGS]
assert len(_src_names) == len(set(_src_names))
print(f"✓ Loaded {len(TABLE_CONFIGS)} table configs")


# %%
# =============================================================================
# CELL 3 — CÁC HÀM TIỆN ÍCH
# =============================================================================

def table_exists(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def require_table(table_name: str) -> None:
    if not table_exists(table_name):
        raise ValueError(f"❌ Không tìm thấy bảng: {table_name}")

def _key_cols(config: dict) -> list:
    """Trả về key columns dạng list — hỗ trợ cả single key (str) và composite key (list)."""
    k = config["key"]
    return k if isinstance(k, list) else [k]

def ensure_target_table_exists(config: dict) -> None:
    """Tự động kiểm tra và tạo mới bảng Delta nếu chưa tồn tại với Liquid Clustering & Auto-optimize."""
    target_table = config["target_table"]
    if not table_exists(target_table):
        parts = target_table.split(".")
        if len(parts) > 1:
            schema_name = ".".join(parts[:-1])
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

        col_defs = []
        for col_name, col_cfg in config["columns"].items():
            ctype = col_cfg["type"].upper()
            if ctype == "EPOCH_SECONDS_TIMESTAMP":
                ctype = "TIMESTAMP"
            col_defs.append(f"    `{col_name}` {ctype}")
        
        col_defs.append("    `deleted` BOOLEAN")
        col_defs.append("    `_ingested_at` TIMESTAMP")
        col_defs.append("    `_source_db` STRING")
        
        cols_sql = ",\n".join(col_defs)
        
        key_cols = _key_cols(config)
        cluster_sql = ", ".join(f"`{k}`" for k in key_cols)
        
        ddl = f"""CREATE TABLE IF NOT EXISTS {target_table} (
{cols_sql}
)
USING DELTA
CLUSTER BY ({cluster_sql})
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)"""
        print(f"🔨 Bảng '{target_table}' chưa tồn tại. Đang tự động tạo mới bảng Delta...")
        spark.sql(ddl)
        print(f"✓ Đã tạo thành công bảng Delta: {target_table}")

def get_table_columns(table_name: str) -> set:
    return set(spark.table(table_name).columns)

def json_value(json_col: str, field_path: str):
    return F.get_json_object(F.col(json_col), f"$.{field_path}")

def cast_json_value(value_col, target_type: str):
    t = target_type.lower()
    if t == "epoch_seconds_timestamp":
        return F.to_timestamp(F.from_unixtime(value_col.cast("bigint")))
    return value_col.cast(t)

def safe_str(value, max_len: int = 4000):
    return None if value is None else str(value)[:max_len]

def _entity_id_expr():
    """Per-table entity_id: single key → field value; composite → concat_ws('|', key fields).
    Reads TABLE_CONFIGS to build a chained WHEN expression — handles both $.id and composite keys.
    concat_ws returns "" when all inputs are null → treated as null via WHEN guard.
    """
    result = F.lit(None).cast("string")
    for cfg in TABLE_CONFIGS:
        keys  = cfg["key"] if isinstance(cfg["key"], list) else [cfg["key"]]
        paths = [cfg["columns"][k]["path"] for k in keys]
        if len(paths) == 1:
            val = F.coalesce(
                F.get_json_object(F.col("after"),  f"$.{paths[0]}"),
                F.get_json_object(F.col("before"), f"$.{paths[0]}"),
            )
        else:
            after_c  = F.concat_ws("|", *[F.get_json_object(F.col("after"),  f"$.{p}") for p in paths])
            before_c = F.concat_ws("|", *[F.get_json_object(F.col("before"), f"$.{p}") for p in paths])
            val = F.coalesce(
                F.when(after_c  != "", after_c),
                F.when(before_c != "", before_c),
            )
        result = F.when(F.col("_source_table") == cfg["source_table"], val).otherwise(result)
    return result

print("✓ Helper functions ready")


# %%
# =============================================================================
# CELL 4 — KIỂM TRA BẢNG TỒN TẠI + ĐỌC WATERMARK
# =============================================================================

for tbl in [RAW_TABLE, WATERMARK_TABLE, RUN_AUDIT_TABLE,
            TABLE_AUDIT_TABLE, REJECT_TABLE, CDC_STATE_TABLE]:
    require_table(tbl)
    print(f"✓ {tbl}")

print("\n--- KIỂM TRA & TẠO BẢNG ĐÍCH (TARGET TABLES) ---")
for cfg in TABLE_CONFIGS:
    ensure_target_table_exists(cfg)

raw_cols = get_table_columns(RAW_TABLE)
for col_name in ["before", "after", "source", "op", RAW_INGEST_COLUMN]:
    if col_name not in raw_cols:
        raise ValueError(f"❌ Cột '{col_name}' không có trong {RAW_TABLE}")

HAS_RAW_SEQUENCE = (RAW_SEQUENCE_COLUMN is not None
                    and RAW_SEQUENCE_COLUMN in raw_cols)
print(f"\nHAS_RAW_SEQUENCE : {HAS_RAW_SEQUENCE}")
print(f"RAW_INGEST_COLUMN: {RAW_INGEST_COLUMN}")

wm_row = (
    spark.table(WATERMARK_TABLE)
    .filter(F.col("watermark_id") == WATERMARK_ID)
    .select("watermark_column", "watermark_value", "watermark_sequence")
    .first()
)
if wm_row is None:
    raise ValueError(f"❌ Không tìm thấy watermark_id = '{WATERMARK_ID}'")

last_watermark = wm_row["watermark_value"] or DEFAULT_WATERMARK
last_sequence  = int(wm_row["watermark_sequence"] or DEFAULT_SEQUENCE)

print(f"\nLast watermark : {last_watermark}")
print(f"Last sequence  : {last_sequence}")


# %%
# =============================================================================
# CELL 5 — XÁC ĐỊNH UPPER BOUND + TẠO RUN AUDIT
# =============================================================================

run_started_at = datetime.utcnow()
raw_base       = spark.table(RAW_TABLE)

# Tính upper_watermark
if HAS_RAW_SEQUENCE:
    ub_row = (
        raw_base
        .select(
            F.col(RAW_INGEST_COLUMN).alias("ts"),
            F.col(RAW_SEQUENCE_COLUMN).cast("bigint").alias("seq"),
        )
        .filter(F.col("ts").isNotNull())
        .orderBy(F.col("ts").desc(), F.col("seq").desc_nulls_last())
        .limit(1)
        .first()
    )
    upper_watermark = ub_row["ts"]       if ub_row else last_watermark
    upper_sequence  = int(ub_row["seq"]) if (ub_row and ub_row["seq"] is not None) else DEFAULT_SEQUENCE
else:
    ub_row = raw_base.agg(F.max(F.col(RAW_INGEST_COLUMN)).alias("ts")).first()
    upper_watermark = ub_row["ts"] if (ub_row and ub_row["ts"] is not None) else last_watermark
    upper_sequence  = DEFAULT_SEQUENCE

print(f"Upper watermark: {upper_watermark}")
print(f"Upper sequence : {upper_sequence}")

run_audit_schema = StructType([
    StructField("run_id",                StringType(),    True),
    StructField("pipeline_name",         StringType(),    True),
    StructField("source_table_name",     StringType(),    True),
    StructField("watermark_id",          StringType(),    True),
    StructField("watermark_from",        TimestampType(), True),
    StructField("watermark_to",          TimestampType(), True),
    StructField("sequence_from",         LongType(),      True),
    StructField("sequence_to",           LongType(),      True),
    StructField("raw_rows_read",         LongType(),      True),
    StructField("valid_rows",            LongType(),      True),
    StructField("deduplicated_rows",     LongType(),      True),
    StructField("rejected_rows",         LongType(),      True),
    StructField("target_table_count",    IntegerType(),   True),
    StructField("succeeded_table_count", IntegerType(),   True),
    StructField("failed_table_count",    IntegerType(),   True),
    StructField("status",                StringType(),    True),
    StructField("started_at",            TimestampType(), True),
    StructField("completed_at",          TimestampType(), True),
    StructField("error_message",         StringType(),    True),
])

spark.createDataFrame(
    [(
        RUN_ID, PIPELINE_NAME, RAW_TABLE, WATERMARK_ID,
        last_watermark, upper_watermark,
        int(last_sequence), int(upper_sequence),
        0, 0, 0, 0,
        TARGET_TABLE_COUNT, 0, 0,
        "RUNNING", run_started_at, None, None,
    )],
    schema=run_audit_schema,
).write.format("delta").mode("append").saveAsTable(RUN_AUDIT_TABLE)

print(f"\n✓ Run audit created — RUN_ID: {RUN_ID}")


# %%
# =============================================================================
# CELL 6 — ĐỌC RAW DATA INCREMENTAL
#
# THAY ĐỔI v4:
#   - KHÔNG persist raw_incremental (quá nặng với 500k+ rows)
#   - Chỉ persist latest_valid (đã dedup, nhỏ hơn nhiều)
#   - Pipeline raw → validate → dedup là 1 chain lazy, Spark chỉ chạy 1 lần
#     khi gặp persist() ở cuối Cell 7
# =============================================================================

effective_lower = last_watermark - timedelta(minutes=WATERMARK_OVERLAP_MINUTES)
lower_cond      = F.col(RAW_INGEST_COLUMN) > F.lit(effective_lower)

if HAS_RAW_SEQUENCE:
    upper_cond = (
        (F.col(RAW_INGEST_COLUMN) < F.lit(upper_watermark))
        | (
            (F.col(RAW_INGEST_COLUMN) == F.lit(upper_watermark))
            & (F.col(RAW_SEQUENCE_COLUMN).cast("bigint") <= F.lit(upper_sequence))
        )
    )
else:
    upper_cond = F.col(RAW_INGEST_COLUMN) <= F.lit(upper_watermark)

raw_seq_col = (
    F.col(RAW_SEQUENCE_COLUMN).cast("bigint")
    if HAS_RAW_SEQUENCE
    else F.lit(DEFAULT_SEQUENCE).cast("bigint")
)

# Lazy DataFrame — chưa thực thi, chỉ định nghĩa kế hoạch đọc
raw_incremental = (
    spark.table(RAW_TABLE)
    .filter(lower_cond & upper_cond)
    .withColumn("_source_table",    F.get_json_object("source", "$.table"))
    .withColumn("_source_db",       F.get_json_object("source", "$.db"))
    .withColumn("_cdc_ts_ms",       F.get_json_object("source", "$.ts_ms").cast("bigint"))
    .withColumn("_cdc_lsn",         F.get_json_object("source", "$.lsn").cast("bigint"))
    .withColumn("_entity_id",       _entity_id_expr())
    .withColumn("_raw_ingested_at", F.col(RAW_INGEST_COLUMN).cast("timestamp"))
    .withColumn("_raw_sequence",    raw_seq_col)
)

print("✓ Raw incremental plan defined (lazy — chưa đọc data)")


# %%
# =============================================================================
# CELL 7 — VALIDATE + DEDUP + PERSIST (1 PASS DUY NHẤT)
#
# THAY ĐỔI v4:
#   - Validate + dedup + persist thành 1 chain duy nhất
#   - Spark đọc raw table đúng 1 lần duy nhất khi gặp persist()
#   - Reject được collect() nhỏ gọn → write riêng sau
#   - raw_rows_read tính từ latest_valid + rejected (tránh scan thêm)
# =============================================================================

known_tables     = [c["source_table"] for c in TABLE_CONFIGS]
# TOAST tables cần ALL events (không dedup) để sequential merge hoạt động đúng.
# Global dedup (rank=1) sẽ loại bỏ event giữa (e.g. event 3 có full JSON) và chỉ
# giữ event cuối (e.g. event 4 có sentinel), khiến TOAST guard không có giá trị thật
# để fallback — dữ liệu bị mất dù Bronze có CASE WHEN sentinel THEN t.col.
toast_tables_set = {c["source_table"] for c in TABLE_CONFIGS if c.get("toast_columns")}

# Validate — persist validated để tránh scan bảng raw 3 lần
validated = raw_incremental.withColumn(
    "_reject_reason",
    F.when(F.col("op").isNull() | (~F.col("op").isin(VALID_OPS)), "INVALID_OPERATION")
     .when(F.col("_source_table").isNull(), "MISSING_SOURCE_TABLE")
     .when(~F.col("_source_table").isin(known_tables), "UNKNOWN_TABLE")
     .when(F.col("_entity_id").isNull() | (F.trim(F.col("_entity_id")) == ""), "MISSING_ENTITY_ID")
     .when(F.col("_cdc_ts_ms").isNull(), "MISSING_EVENT_ORDER")
     .when(F.col("_raw_ingested_at").isNull(), "MISSING_RAW_CURSOR")
).persist(StorageLevel.MEMORY_AND_DISK)

# Dedup window — chỉ dùng cho non-TOAST tables
dedup_window = Window.partitionBy("_source_table", "_entity_id").orderBy(
    F.col("_cdc_ts_ms").desc(),
    F.col("_cdc_lsn").desc_nulls_last(),
    F.col("_raw_ingested_at").desc(),
    F.col("_raw_sequence").desc_nulls_last(),
    F.when(F.col("op") == "d", 1).otherwise(0).desc(),
)

# Non-TOAST: dedup → 1 event/entity (nhỏ, persist hiệu quả)
latest_valid = (
    validated
    .filter(F.col("_reject_reason").isNull())
    .filter(~F.col("_source_table").isin(list(toast_tables_set)))
    .drop("_reject_reason")
    .withColumn("_rank", F.row_number().over(dedup_window))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# TOAST: giữ ALL events — sequential merge trong Cell 10 cần đủ events theo thứ tự
all_valid_toast = (
    validated
    .filter(F.col("_reject_reason").isNull())
    .filter(F.col("_source_table").isin(list(toast_tables_set)))
    .drop("_reject_reason")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

non_toast_count   = latest_valid.count()
toast_count       = all_valid_toast.count()
valid_rows        = non_toast_count + toast_count
deduplicated_rows = valid_rows
print(f"✓ Non-TOAST (deduped): {non_toast_count:,}  |  TOAST (all events): {toast_count:,}")
print(f"✓ Total valid rows   : {valid_rows:,}")

# Reject: lọc từ validated đã cached
reject_df = validated.filter(F.col("_reject_reason").isNotNull())
rejected_rows = reject_df.count()
raw_rows_read = valid_rows + rejected_rows   # ước tính tổng

if rejected_rows > 0:
    reject_df.select(
        F.lit(RUN_ID).alias("run_id"),
        F.lit(PIPELINE_NAME).alias("pipeline_name"),
        F.col("_source_table").alias("source_table_name"),
        F.col("_source_db").alias("source_db"),
        "op",
        F.col("_entity_id").alias("entity_id"),
        F.col("_cdc_ts_ms").alias("source_ts_ms"),
        F.col("_cdc_lsn").alias("source_lsn"),
        F.col("_raw_sequence").alias("raw_sequence"),
        F.col("_raw_ingested_at").alias("raw_ingested_at"),
        F.col("before").alias("before_payload"),
        F.col("after").alias("after_payload"),
        F.col("source").alias("source_payload"),
        F.col("_reject_reason").alias("reject_reason"),
        F.lit(None).cast("string").alias("reject_detail"),
        F.current_timestamp().alias("rejected_at"),
    ).write.format("delta").mode("append").saveAsTable(REJECT_TABLE)
    print(f"⚠️  Rejected: {rejected_rows:,} rows")

validated.unpersist()
print(f"✓ Raw rows read (approx): {raw_rows_read:,}")


# %%
# =============================================================================
# CELL 8 — LOAD CDC_RECORD_STATE THÀNH SPARK DATAFRAME
#
# THAY ĐỔI v4:
#   - Không collect() về Python dict nữa
#   - Giữ state là Spark DataFrame → native join trong Cell 10
#   - Tránh Python UDF (bottleneck lớn nhất với 500k rows)
#   - DataFrame nhỏ (chỉ entity_id + ordering cols) → broadcast join tự động
# =============================================================================

# state_df: DataFrame với các cột cần để so sánh thứ tự event
state_df = (
    spark.table(CDC_STATE_TABLE)
    .filter(F.col("source_table_name").isin(known_tables))
    .select(
        "source_table_name",
        "entity_id",
        F.col("cdc_ts_ms").alias("st_cdc_ts_ms"),
        F.col("cdc_lsn").alias("st_cdc_lsn"),
        F.col("raw_ingested_at").alias("st_raw_ingested_at"),
        F.col("raw_sequence").alias("st_raw_sequence"),
    )
    # Cache vì 16 bảng đều join với nó
    .cache()
)

total_state_rows = state_df.count()
print(f"✓ State loaded: {total_state_rows:,} records (Spark DataFrame, cached)")


# %%
# =============================================================================
# CELL 9 — HÀM BUILD SOURCE (TOAST-AWARE)
# Non-TOAST: đọc từ latest_valid (1 event/entity, Cell 7)
# TOAST    : đọc từ all_valid_toast (ALL events, Cell 7) → sequential merge đúng thứ tự
# =============================================================================

def build_events_for_table(config: dict) -> DataFrame:
    """
    Non-TOAST: đọc từ latest_valid (1 event/entity) → 1 MERGE
    TOAST    : đọc từ all_valid_toast (ALL events) → sắp xếp ts ASC → sequential merge
               Đảm bảo event giữa (có giá trị thật) được merge trước event cuối (có sentinel).
    """
    has_toast = len(config.get("toast_columns", [])) > 0
    source_df = all_valid_toast if has_toast else latest_valid

    col_exprs = []
    for col_name, col_cfg in config["columns"].items():
        raw_val = F.when(
            F.col("op") == "d",
            json_value("before", col_cfg["path"])
        ).otherwise(
            json_value("after", col_cfg["path"])
        )
        col_exprs.append(cast_json_value(raw_val, col_cfg["type"]).alias(col_name))

    base = (
        source_df
        .filter(F.col("_source_table") == config["source_table"])
        .select(
            *col_exprs,
            F.col("op").alias("_cdc_op"),
            "_cdc_ts_ms", "_cdc_lsn",
            "_raw_ingested_at", "_raw_sequence", "_source_db",
            F.when(F.col("op") == "d", True).otherwise(False).alias("deleted"),
            F.current_timestamp().alias("_processed_at"),
            F.lit(RUN_ID).alias("_last_run_id"),
        )
        .filter(_key_filter(config))
    )

    if not has_toast:
        dedup_w = Window.partitionBy(*_key_cols(config)).orderBy(
            F.col("_cdc_ts_ms").desc(),
            F.col("_cdc_lsn").desc_nulls_last(),
            F.col("_raw_ingested_at").desc(),
            F.col("_raw_sequence").desc_nulls_last(),
            F.when(F.col("_cdc_op") == "d", 1).otherwise(0).desc(),
        )
        return (
            base
            .withColumn("_rank", F.row_number().over(dedup_w))
            .filter(F.col("_rank") == 1)
            .drop("_rank")
            .orderBy(F.col("_cdc_ts_ms").asc_nulls_last())
        )
    else:
        return base.orderBy(
            F.col("_cdc_ts_ms").asc_nulls_last(),
            F.col("_cdc_lsn").asc_nulls_last(),
        )

def is_toast_table(config: dict) -> bool:
    return len(config.get("toast_columns", [])) > 0

print("✓ Cell 9: build_events_for_table ready")


# %%
# =============================================================================
# CELL 10 — MERGE TARGET (TOAST SEQUENTIAL MERGE)
# =============================================================================

def _key_filter(config: dict):
    """Filter isNotNull cho tất cả key columns."""
    cols = _key_cols(config)
    cond = F.col(cols[0]).isNotNull()
    for c in cols[1:]:
        cond = cond & F.col(c).isNotNull()
    return cond

def _key_entity_expr(config: dict):
    """Biểu thức entity_id: single key → cast string; composite → concat_ws('|', ...)."""
    cols = _key_cols(config)
    if len(cols) == 1:
        return F.col(cols[0]).cast("string")
    return F.concat_ws("|", *[F.col(c).cast("string") for c in cols])

def _merge_condition(config: dict) -> str:
    """MERGE ON condition: single → t.k = s.k; composite → AND của tất cả keys."""
    return " AND ".join(f"t.`{k}` = s.`{k}`" for k in _key_cols(config))


def build_upsert_set(config: dict) -> dict:
    """
    WHEN MATCHED UPDATE:
    TOAST columns: giữ nguyên giá trị cũ (t.col) khi source là sentinel HOẶC NULL.
    Đảm bảo giá trị thật đã có trong bảng không bị ghi đè bởi sentinel.
    Non-TOAST: luôn update bình thường.
    """
    toast_cols = set(config.get("toast_columns", []))
    mapping    = {}
    key_set = set(_key_cols(config))
    for col_name in config["columns"].keys():
        if col_name in key_set:
            continue
        if col_name in toast_cols:
            mapping[col_name] = (
                f"CASE WHEN s.`{col_name}` = '{TOAST_SENTINEL}' "
                f"     OR s.`{col_name}` IS NULL "
                f"THEN t.`{col_name}` "
                f"ELSE s.`{col_name}` END"
            )
        else:
            mapping[col_name] = f"s.`{col_name}`"
    mapping["deleted"]      = "false"
    mapping["_ingested_at"] = "current_timestamp()"
    mapping["_source_db"]   = "s.`_source_db`"
    return mapping


def build_insert_values(config: dict) -> dict:
    """TOAST sentinel → NULL khi INSERT mới (không có giá trị thật)."""
    toast_cols = set(config.get("toast_columns", []))
    mapping    = {}
    for col_name in config["columns"].keys():
        if col_name in toast_cols:
            mapping[col_name] = (
                f"CASE WHEN s.`{col_name}` = '{TOAST_SENTINEL}' "
                f"THEN NULL ELSE s.`{col_name}` END"
            )
        else:
            mapping[col_name] = f"s.`{col_name}`"
    mapping["deleted"]      = "false"
    mapping["_ingested_at"] = "current_timestamp()"
    mapping["_source_db"]   = "s.`_source_db`"
    return mapping


def merge_single_event(event_df: DataFrame, config: dict) -> None:
    (
        DeltaTable.forName(spark, config["target_table"]).alias("t")
        .merge(event_df.alias("s"), _merge_condition(config))
        .whenMatchedUpdate(condition="s._cdc_op = 'd'",
                           set={"deleted": "true"})
        .whenMatchedUpdate(condition="s._cdc_op IN ('c','r','u')",
                           set=build_upsert_set(config))
        .whenNotMatchedInsert(condition="s._cdc_op IN ('c','r','u')",
                               values=build_insert_values(config))
        .execute()
    )


def filter_stale_native(events_df: DataFrame, config: dict) -> tuple[DataFrame, int]:
    """
    Lọc stale events dùng Spark native join thay Python UDF.

    Logic:
      1. Join events với state theo (source_table_name, entity_id)
      2. LEFT JOIN → record chưa có trong state → applicable
      3. So sánh struct ordering: event > state → applicable
      4. Bỏ qua: event <= state → stale
    """
    source_table = config["source_table"]

    table_state_df = state_df.filter(
        F.col("source_table_name") == source_table
    ).select(
        "entity_id",
        "st_cdc_ts_ms", "st_cdc_lsn",
        "st_raw_ingested_at", "st_raw_sequence",
    )

    if table_state_df.isEmpty():
        return events_df, 0

    latest_w = Window.partitionBy(*_key_cols(config)).orderBy(
        F.col("_cdc_ts_ms").desc(),
        F.col("_cdc_lsn").desc_nulls_last(),
    )

    events_ranked = (
        events_df
        .withColumn("_ev_rank",    F.row_number().over(latest_w))
        .withColumn("_entity_id",  _key_entity_expr(config))
    )

    applicable_keys = (
        events_ranked
        .filter(F.col("_ev_rank") == 1)
        .join(
            table_state_df,
            events_ranked["_entity_id"] == table_state_df["entity_id"],
            "left",
        )
        .filter(
            F.col("entity_id").isNull()
            | (
                F.struct(
                    F.coalesce("_cdc_ts_ms",      F.lit(-1).cast("bigint")),
                    F.coalesce("_cdc_lsn",        F.lit(-1).cast("bigint")),
                    F.coalesce("_raw_ingested_at", F.lit(DEFAULT_WATERMARK).cast("timestamp")),
                    F.coalesce("_raw_sequence",    F.lit(-1).cast("bigint")),
                )
                >
                F.struct(
                    F.coalesce("st_cdc_ts_ms",      F.lit(-1).cast("bigint")),
                    F.coalesce("st_cdc_lsn",        F.lit(-1).cast("bigint")),
                    F.coalesce("st_raw_ingested_at", F.lit(DEFAULT_WATERMARK).cast("timestamp")),
                    F.coalesce("st_raw_sequence",    F.lit(-1).cast("bigint")),
                )
            )
        )
        .select(F.col("_entity_id").alias("_applicable_key"))
        .distinct()
    )

    result = (
        events_ranked
        .drop("_ev_rank")
        .join(
            applicable_keys,
            events_ranked["_entity_id"] == applicable_keys["_applicable_key"],
            "inner",
        )
        .drop("_applicable_key", "_entity_id")
    )

    applicable_count = result.count()
    return result, applicable_count


def _make_audit_row(source_table, target_table, input_rows, dedup_rows,
                    applicable_rows, upsert_rows, delete_rows, skipped,
                    started_at, status="SUCCESS", error_message=None) -> dict:
    return {
        "run_id":               RUN_ID,
        "pipeline_name":        PIPELINE_NAME,
        "source_table_name":    source_table,
        "target_table_name":    target_table,
        "input_rows":           input_rows,
        "deduplicated_rows":    dedup_rows,
        "applicable_rows":      applicable_rows,
        "upsert_event_rows":    upsert_rows,
        "delete_event_rows":    delete_rows,
        "skipped_stale_rows":   skipped,
        "rejected_rows":        0,
        "status":               status,
        "started_at":           started_at,
        "completed_at":         datetime.utcnow(),
        "error_message":        error_message,
    }


def process_table(config: dict, state_dfs_out: list, audit_rows_out: list) -> str:
    table_started_at = datetime.utcnow()
    source_table     = config["source_table"]
    target_table     = config["target_table"]
    has_toast        = is_toast_table(config)

    try:
        events_df  = build_events_for_table(config).persist(StorageLevel.MEMORY_AND_DISK)
        input_rows = events_df.count()

        if input_rows == 0:
            events_df.unpersist()
            audit_rows_out.append(_make_audit_row(
                source_table, target_table,
                input_rows=0, dedup_rows=0, applicable_rows=0,
                upsert_rows=0, delete_rows=0, skipped=0,
                started_at=table_started_at,
            ))
            return "SUCCESS"

        applicable_df, applicable_rows = filter_stale_native(events_df, config)
        skipped = input_rows - applicable_rows

        if applicable_rows == 0:
            events_df.unpersist()
            audit_rows_out.append(_make_audit_row(
                source_table, target_table,
                input_rows=input_rows, dedup_rows=input_rows,
                applicable_rows=0, upsert_rows=0, delete_rows=0, skipped=skipped,
                started_at=table_started_at,
            ))
            return "SUCCESS"

        op_counts = (
            applicable_df
            .groupBy(F.col("_cdc_op").alias("op_type"))
            .count()
            .collect()
        )
        op_map    = {row["op_type"]: row["count"] for row in op_counts}
        upsert_rows = sum(op_map.get(op, 0) for op in UPSERT_OPS)
        delete_rows = op_map.get("d", 0)

        if not has_toast:
            merge_single_event(applicable_df, config)
        else:
            resolve_w = (
                Window.partitionBy(*_key_cols(config))
                .orderBy(
                    F.col("_cdc_ts_ms").asc_nulls_last(),
                    F.col("_cdc_lsn").asc_nulls_last(),
                )
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )

            resolved_df = applicable_df
            for col_name in config["toast_columns"]:
                resolved_df = resolved_df.withColumn(
                    col_name,
                    F.last(
                        F.when(
                            F.col(col_name).isNotNull() & (F.col(col_name) != TOAST_SENTINEL),
                            F.col(col_name),
                        ),
                        ignorenulls=True,
                    ).over(resolve_w),
                )

            latest_ev_w = Window.partitionBy(*_key_cols(config)).orderBy(
                F.col("_cdc_ts_ms").desc(),
                F.col("_cdc_lsn").desc_nulls_last(),
            )
            applicable_df = (
                resolved_df
                .withColumn("_r", F.row_number().over(latest_ev_w))
                .filter(F.col("_r") == 1)
                .drop("_r")
                .persist(StorageLevel.MEMORY_AND_DISK)
            )

            merge_single_event(applicable_df, config)

        latest_w2 = Window.partitionBy(*_key_cols(config)).orderBy(
            F.col("_cdc_ts_ms").desc(),
            F.col("_cdc_lsn").desc_nulls_last(),
        )
        table_state_df = (
            applicable_df
            .withColumn("_r", F.row_number().over(latest_w2))
            .filter(F.col("_r") == 1)
            .select(
                F.lit(source_table).alias("source_table_name"),
                F.lit(target_table).alias("target_table_name"),
                _key_entity_expr(config).alias("entity_id"),
                F.col("_cdc_op").alias("cdc_op"),
                F.col("_cdc_ts_ms").alias("cdc_ts_ms"),
                F.col("_cdc_lsn").alias("cdc_lsn"),
                F.col("_raw_sequence").alias("raw_sequence"),
                F.col("_raw_ingested_at").alias("raw_ingested_at"),
                F.col("deleted").alias("is_deleted"),
                F.col("_processed_at").alias("processed_at"),
                F.col("_last_run_id").alias("last_run_id"),
            )
        )
        state_dfs_out.append(table_state_df)

        merge_type = "pre-resolve(TOAST)" if has_toast else "single"
        print(f"         merge={merge_type}  applicable={applicable_rows}  skipped={skipped}")

        if has_toast:
            applicable_df.unpersist()
        events_df.unpersist()

        audit_rows_out.append(_make_audit_row(
            source_table, target_table,
            input_rows=input_rows, dedup_rows=input_rows,
            applicable_rows=applicable_rows,
            upsert_rows=upsert_rows, delete_rows=delete_rows, skipped=skipped,
            started_at=table_started_at,
        ))
        return "SUCCESS"

    except Exception as e:
        try:
            events_df.unpersist()
        except Exception:
            pass
        audit_rows_out.append(_make_audit_row(
            source_table, target_table,
            input_rows=0, dedup_rows=0, applicable_rows=0,
            upsert_rows=0, delete_rows=0, skipped=0,
            started_at=table_started_at,
            status="FAILED",
            error_message=safe_str(traceback.format_exc()),
        ))
        raise


print("✓ Cell 10: process_table ready")

# %%
# CELL 11 — VÒNG LẶP TUẦN TỰ 16 BẢNG
# =============================================================================

state_dfs_all    = []
audit_rows_all   = []
succeeded_count  = 0
failed_count     = 0
processing_error = None

for idx, config in enumerate(TABLE_CONFIGS, start=1):
    src = config["source_table"]
    tgt = config["target_table"]
    print(f"\n[{idx:02d}/{TARGET_TABLE_COUNT}] {src}")
    print(f"         → {tgt}")

    try:
        process_table(config, state_dfs_all, audit_rows_all)
        succeeded_count += 1
        print(f"         ✓ SUCCESS")
    except Exception as e:
        failed_count    += 1
        processing_error = e
        print(f"         ❌ FAILED: {e}")
        break

print(f"\n{'='*50}")
print(f"Succeeded: {succeeded_count}/{TARGET_TABLE_COUNT}")
print(f"Failed   : {failed_count}")


# %%
# =============================================================================
# CELL 12 — BATCH COMMIT: STATE + AUDIT + WATERMARK
# =============================================================================

from functools import reduce

# Schema cho cdc_table_run_audit
audit_schema = StructType([
    StructField("run_id",             StringType(),    True),
    StructField("pipeline_name",      StringType(),    True),
    StructField("source_table_name",  StringType(),    True),
    StructField("target_table_name",  StringType(),    True),
    StructField("input_rows",         LongType(),      True),
    StructField("deduplicated_rows",  LongType(),      True),
    StructField("applicable_rows",    LongType(),      True),
    StructField("upsert_event_rows",  LongType(),      True),
    StructField("delete_event_rows",  LongType(),      True),
    StructField("skipped_stale_rows", LongType(),      True),
    StructField("rejected_rows",      LongType(),      True),
    StructField("status",             StringType(),    True),
    StructField("started_at",         TimestampType(), True),
    StructField("completed_at",       TimestampType(), True),
    StructField("error_message",      StringType(),    True),
])

run_completed_at = datetime.utcnow()

try:
    if processing_error is not None:
        # ---- FAILED ----
        DeltaTable.forName(spark, RUN_AUDIT_TABLE).update(
            F.col("run_id") == F.lit(RUN_ID),
            {
                "status":                F.lit("FAILED"),
                "completed_at":          F.lit(run_completed_at),
                "raw_rows_read":         F.lit(raw_rows_read),
                "valid_rows":            F.lit(valid_rows),
                "deduplicated_rows":     F.lit(deduplicated_rows),
                "rejected_rows":         F.lit(rejected_rows),
                "succeeded_table_count": F.lit(succeeded_count),
                "failed_table_count":    F.lit(failed_count),
                "error_message":         F.lit(safe_str(str(processing_error))),
            },
        )

        if audit_rows_all:
            audit_tuples = [
                (
                    r["run_id"], r["pipeline_name"],
                    r["source_table_name"], r["target_table_name"],
                    r["input_rows"], r["deduplicated_rows"],
                    r["applicable_rows"], r["upsert_event_rows"],
                    r["delete_event_rows"], r["skipped_stale_rows"],
                    r["rejected_rows"], r["status"],
                    r["started_at"], r["completed_at"],
                    r["error_message"],
                )
                for r in audit_rows_all
            ]
            spark.createDataFrame(audit_tuples, schema=audit_schema).write \
                .format("delta").mode("append").saveAsTable(TABLE_AUDIT_TABLE)

        DeltaTable.forName(spark, WATERMARK_TABLE).update(
            F.col("watermark_id") == F.lit(WATERMARK_ID),
            {
                "status":        F.lit("FAILED"),
                "flow_name":     F.lit(PIPELINE_NAME),
                "last_run_id":   F.lit(RUN_ID),
                "error_message": F.lit(safe_str(str(processing_error))),
                "update_time":   F.current_timestamp(),
            },
        )
        print(f"❌ Run FAILED — watermark KHÔNG advance")
        raise processing_error

    # ---- SUCCESS ----

    # 1. MERGE cdc_record_state — 1 lần sau cùng (Pure Spark Engine, không qua Python Driver)
    if state_dfs_all:
        state_df_out = reduce(DataFrame.unionByName, state_dfs_all)
        (
            DeltaTable.forName(spark, CDC_STATE_TABLE).alias("t")
            .merge(
                state_df_out.alias("s"),
                "t.source_table_name = s.source_table_name "
                "AND t.entity_id = s.entity_id",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✓ cdc_record_state: MERGE state across all tables")

    # 2. INSERT audit — 1 lần
    if audit_rows_all:
        audit_tuples = [
            (
                r["run_id"], r["pipeline_name"],
                r["source_table_name"], r["target_table_name"],
                r["input_rows"], r["deduplicated_rows"],
                r["applicable_rows"], r["upsert_event_rows"],
                r["delete_event_rows"], r["skipped_stale_rows"],
                r["rejected_rows"], r["status"],
                r["started_at"], r["completed_at"],
                r["error_message"],
            )
            for r in audit_rows_all
        ]
        spark.createDataFrame(audit_tuples, schema=audit_schema).write \
            .format("delta").mode("append").saveAsTable(TABLE_AUDIT_TABLE)
        print(f"✓ cdc_table_run_audit: INSERT {len(audit_rows_all)} rows")

    # 3. UPDATE run audit = SUCCESS
    DeltaTable.forName(spark, RUN_AUDIT_TABLE).update(
        F.col("run_id") == F.lit(RUN_ID),
        {
            "status":                F.lit("SUCCESS"),
            "completed_at":          F.lit(run_completed_at),
            "raw_rows_read":         F.lit(raw_rows_read),
            "valid_rows":            F.lit(valid_rows),
            "deduplicated_rows":     F.lit(deduplicated_rows),
            "rejected_rows":         F.lit(rejected_rows),
            "succeeded_table_count": F.lit(succeeded_count),
            "failed_table_count":    F.lit(0),
            "error_message":         F.lit(None).cast("string"),
        },
    )
    print(f"✓ cdc_run_audit: UPDATE SUCCESS")

    # 4. Advance watermark — CHỈ khi toàn bộ SUCCESS
    DeltaTable.forName(spark, WATERMARK_TABLE).update(
        F.col("watermark_id") == F.lit(WATERMARK_ID),
        {
            "watermark_value":    F.lit(upper_watermark),
            "watermark_sequence": F.lit(upper_sequence),
            "status":             F.lit("SUCCESS"),
            "flow_name":          F.lit(PIPELINE_NAME),
            "last_run_id":        F.lit(RUN_ID),
            "error_message":      F.lit(None).cast("string"),
            "update_time":        F.current_timestamp(),
        },
    )
    print(f"✓ brz_watermark: advance → {upper_watermark}")

    print(f"\n{'='*50}")
    print(f"✅ RUN SUCCESS — RUN_ID: {RUN_ID}")
    print(f"   Rows read  : {raw_rows_read:,}")
    print(f"   Valid rows : {valid_rows:,}")
    print(f"   Rejected   : {rejected_rows:,}")
    print(f"   Tables OK  : {succeeded_count}/{TARGET_TABLE_COUNT}")
    print(f"   Watermark →: {upper_watermark}")

finally:
    # Giải phóng bộ nhớ dù thành công hay thất bại
    latest_valid.unpersist()
    all_valid_toast.unpersist()
    state_df.unpersist()
    try:
        validated.unpersist()
    except Exception:
        pass
    print("\n✓ Unpersisted DataFrames")


# =============================================================================
# QUERY KIỂM TRA (chạy trong cell SQL riêng)
# =============================================================================
# SELECT run_id, status, raw_rows_read, valid_rows, rejected_rows,
#        succeeded_table_count, failed_table_count, started_at, completed_at
# FROM lh_vv_bronze.partner.cdc_run_audit ORDER BY started_at DESC LIMIT 5;
#
# SELECT source_table_name, status, input_rows, applicable_rows,
#        upsert_event_rows, delete_event_rows, skipped_stale_rows
# FROM lh_vv_bronze.partner.cdc_table_run_audit
# WHERE run_id = '<RUN_ID>' ORDER BY started_at;
#
# SELECT source_table_name, entity_id, COUNT(*) cnt
# FROM lh_vv_bronze.partner.cdc_record_state
# GROUP BY source_table_name, entity_id HAVING cnt > 1;
# -- Kết quả mong đợi: 0 dòng
#
# SELECT watermark_id, watermark_value, status, update_time
# FROM lh_vv_bronze.partner.brz_watermark
# WHERE watermark_id = 'wm_transform_partner_raw_data';
# =============================================================================

StatementMeta(, 13c1afe3-343b-42b1-a432-1c186420413e, 3, Finished, Available, Finished, False)

PIPELINE         : transform_partner_raw_data_v2
RUN_ID           : a3e6e776-75a6-451f-90cd-3069e717f420
RAW TABLE        : lh_vv_bronze.dbo.partner_raw_data
RAW_INGEST_COLUMN: EventProcessedUtcTime
WATERMARK TABLE  : lh_vv_bronze.partner.brz_watermark
✓ Loaded 18 table configs
✓ Helper functions ready
✓ lh_vv_bronze.dbo.partner_raw_data
✓ lh_vv_bronze.partner.brz_watermark
✓ lh_vv_bronze.partner.cdc_run_audit
✓ lh_vv_bronze.partner.cdc_table_run_audit
✓ lh_vv_bronze.partner.partner_cdc_reject
✓ lh_vv_bronze.partner.cdc_record_state

HAS_RAW_SEQUENCE : False
RAW_INGEST_COLUMN: EventProcessedUtcTime

Last watermark : 2026-07-21 01:52:40.819839
Last sequence  : -1
Upper watermark: 2026-08-18 07:03:44.939251
Upper sequence : -1

✓ Run audit created — RUN_ID: a3e6e776-75a6-451f-90cd-3069e717f420
✓ Raw incremental plan defined (lazy — chưa đọc data)
✓ Non-TOAST (deduped): 301  |  TOAST (all events): 275,110
✓ Total valid rows   : 275,411
⚠️  Rejected: 9,022 rows
✓ Raw rows read (approx): 28

In [3]:
%%sql
SELECT * FROM lh_vv_gold.dbo.poi_master WHERE poi_id = 'c4fe345e-b57d-4b4d-562c-979b4c2a8513';


-- DESCRIBE TABLE lh_vv_gold.dbo.poi_master;

-- DESCRIBE TABLE lh_vv_gold.dbo.poi_detail;


-- SELECT
-- *
-- FROM lh_vv_bronze.dbo.partner_raw_data
-- WHERE get_json_object(after,  '$.id') = '0653ba67-4a06-3629-47e3-3ada78ab82b9'
--    OR get_json_object(before, '$.id') = '0653ba67-4a06-3629-47e3-3ada78ab82b9'
-- ORDER BY EventProcessedUtcTime ASC;



-- ✅ ĐÚNG schema: lh_vv_bronze.partner (không phải .dbo)
-- ✅ ĐÚNG giá trị: 2000-01-01 (không phải 1900-01-01)
-- DELETE FROM lh_vv_bronze.partner.brz_watermark
-- WHERE watermark_id = 'wm_transform_partner_raw_data';

-- INSERT INTO lh_vv_bronze.partner.brz_watermark
-- VALUES (
--     'wm_transform_partner_raw_data',
--     'lh_vv_bronze.dbo.partner_raw_data',
--     'EventProcessedUtcTime',
--     to_timestamp('2026-07-10 00:00:00', 'yyyy-MM-dd HH:mm:ss'),
--     CAST(-1 AS BIGINT),
--     current_timestamp(),
--     'INITIALIZED',
--     'transform_partner_raw_data_v2',
--     NULL,
--     NULL
-- );


-- 2. BRONZE business_services: record sau khi CDC transform
-- SELECT *
-- FROM lh_vv_bronze.dbo.partner_raw_data where source like('%business_service_reviews%');



StatementMeta(, c9d1c1fb-cab6-454f-aebc-f9553187fe4c, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 28 fields>